In [2]:
import time
import win32api
import win32gui
import win32process
import psutil
import pyperclip

def get_active_window_info():
    """取得當前最前景視窗的標題與進程名稱（檔名）"""
    hwnd = win32gui.GetForegroundWindow()
    if not hwnd:
        return "未知視窗", "未知程式"
    
    # 視窗標題（通常包含開啟的檔案名稱，如: report.docx - Word）
    window_title = win32gui.GetWindowText(hwnd)
    
    # 執行檔名稱
    try:
        _, pid = win32process.GetWindowThreadProcessId(hwnd)
        process = psutil.Process(pid)
        process_name = process.name()
    except Exception:
        process_name = "未知程式"
        
    return window_title, process_name

def monitor_clipboard():
    print("剪貼簿監聽中...（按下 Ctrl+C 結束程式）\n" + "-"*40)
    last_text = pyperclip.paste()

    while True:
        try:
            current_text = pyperclip.paste()
            
            # 檢查剪貼簿內容是否有更新
            if current_text != last_text:
                last_text = current_text
                
                # 只有當剪貼簿內容非空白時處理
                if current_text.strip():
                    title, proc = get_active_window_info()
                    print(f"【檢測到複製】")
                    print(f"來源程式: {proc}")
                    print(f"視窗標題: {title}")
                    print(f"複製內容: {current_text[:50]}{'...' if len(current_text) > 50 else ''}")
                    print("-" * 40)
                    
            time.sleep(0.3)  # 輪詢間隔 0.3 秒，降低 CPU 佔用
            
        except KeyboardInterrupt:
            print("\n已停止監聽。")
            break
        except Exception as e:
            time.sleep(0.5)

if __name__ == "__main__":
    monitor_clipboard()

剪貼簿監聽中...（按下 Ctrl+C 結束程式）
----------------------------------------

已停止監聽。
